# Phase 4: Hybrid Recommendation System

In this notebook, we combine the predictive accuracy of SVD with the explainability of Item-Based Collaborative Filtering. By using a weighted average of both models, we aim to get the best of both worlds.

**Hybrid Strategy:**
- SVD Weight: 0.7
- Item-CF Weight: 0.3


In [1]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
from surprise import Dataset, Reader, SVD
from IPython.display import display
import warnings

warnings.filterwarnings('ignore')


## 1. Load Data and Train Models
We load the SVD subset (6.9M ratings) to ensure consistency across both models. Then, we quickly retrain both the SVD and Item-CF models so they are ready for inference.

In [2]:
# Load data
df = pd.read_csv('../data/processed/netflix_svd_subset.csv')
titles_df = pd.read_csv('../data/raw/movie_titles.csv', encoding='ISO-8859-1', header=None, names=['movie_id', 'year', 'title'], on_bad_lines='skip')
title_map = dict(zip(titles_df['movie_id'], titles_df['title']))

# -------------------------
# Train Item-CF Model
# -------------------------
user_ids = df['user_id'].astype('category')
movie_ids = df['movie_id'].astype('category')

user_idx = user_ids.cat.codes
movie_idx = movie_ids.cat.codes

user_map = dict(enumerate(user_ids.cat.categories))
movie_map = dict(enumerate(movie_ids.cat.categories))
rev_user_map = {v: k for k, v in user_map.items()}
rev_movie_map = {v: k for k, v in movie_map.items()}

sparse_user_item = csr_matrix((df['rating'], (user_idx, movie_idx)), shape=(len(user_map), len(movie_map)))
item_similarity = cosine_similarity(sparse_user_item.T)
np.fill_diagonal(item_similarity, 0)

# -------------------------
# Train SVD Model
# -------------------------
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df[['user_id', 'movie_id', 'rating']], reader)
trainset = data.build_full_trainset()
svd_model = SVD(n_factors=50, n_epochs=20, random_state=42)
svd_model.fit(trainset)

print("Models trained successfully!")


Models trained successfully!


## 2. Hybrid Recommendation Function
We combine the scores using `Final Score = 0.7 * SVD + 0.3 * ItemCF`. Since Item-CF dot product scores can be large and SVD outputs 1-5 ratings, we use Min-Max normalization to scale both scores between 0 and 1 before combining them.

In [3]:
def get_item_cf_scores(user_id):
    if user_id not in rev_user_map:
        return np.zeros(len(movie_map)), np.zeros(len(movie_map))
    idx = rev_user_map[user_id]
    user_ratings = sparse_user_item[idx].toarray().flatten()
    scores = item_similarity.dot(user_ratings)
    return scores, user_ratings

def recommend_hybrid(user_id, top_n=10):
    if user_id not in rev_user_map:
        return pd.DataFrame()
        
    all_movies = df['movie_id'].unique()
    
    # Get Item-CF scores and the user's historical ratings
    cf_scores, user_ratings = get_item_cf_scores(user_id)
    
    results = []
    for m_id in all_movies:
        idx = rev_movie_map[m_id]
        
        # Skip movies the user has already watched
        if user_ratings[idx] > 0:
            continue
            
        cf_score = cf_scores[idx]
        svd_score = svd_model.predict(uid=user_id, iid=m_id).est
        
        results.append({
            'movie_id': m_id,
            'title': title_map.get(m_id, f"Unknown (ID: {m_id})"),
            'raw_svd': svd_score,
            'raw_cf': cf_score
        })
        
    res_df = pd.DataFrame(results)
    if res_df.empty:
        return res_df
        
    # Min-Max Normalization to scale both columns between 0 and 1
    min_svd, max_svd = res_df['raw_svd'].min(), res_df['raw_svd'].max()
    min_cf, max_cf = res_df['raw_cf'].min(), res_df['raw_cf'].max()
    
    # Prevent division by zero
    svd_range = (max_svd - min_svd) if (max_svd - min_svd) > 0 else 1
    cf_range = (max_cf - min_cf) if (max_cf - min_cf) > 0 else 1
    
    res_df['norm_svd'] = (res_df['raw_svd'] - min_svd) / svd_range
    res_df['norm_cf'] = (res_df['raw_cf'] - min_cf) / cf_range
    
    # Weighted Average combining both models
    res_df['final_score'] = (0.7 * res_df['norm_svd']) + (0.3 * res_df['norm_cf'])
    
    # Add Explainability
    res_df['explanation'] = "Recommended because similar users liked this movie and it is also similar to movies previously rated highly by the user."
    
    # Sort descending and return the Top N
    res_df = res_df.sort_values(by='final_score', ascending=False).head(top_n)
    
    # Clean up output display
    return res_df[['movie_id', 'title', 'norm_svd', 'norm_cf', 'final_score', 'explanation']].round(3)


## 3. Demonstration
We generate hybrid recommendations for 5 sample users.

In [4]:
# Extract 5 sample users
sample_users = df['user_id'].unique()[:5]

for u in sample_users:
    print(f"--- Top 5 Hybrid Recommendations for User: {u} ---")
    display(recommend_hybrid(u, top_n=5))
    print()


--- Top 5 Hybrid Recommendations for User: 2031561 ---


,movie_id,title,norm_svd,norm_cf,final_score,explanation
2357,6450,City of God,0.992,0.633,0.885,Recommended because similar users liked this m...
1399,3917,Garden State,0.883,0.846,0.872,Recommended because similar users liked this m...
1232,3446,Spirited Away,0.978,0.591,0.862,Recommended because similar users liked this m...
5254,14274,I Heart Huckabees,0.921,0.662,0.843,Recommended because similar users liked this m...
1891,5226,Rushmore,0.843,0.840,0.842,Recommended because similar users liked this m...



--- Top 5 Hybrid Recommendations for User: 701730 ---


,movie_id,title,norm_svd,norm_cf,final_score,explanation
1274,3605,The Wizard of Oz: Collector's Edition,0.913,0.924,0.916,Recommended because similar users liked this m...
2054,5732,GoodFellas: Special Edition,0.879,0.946,0.899,Recommended because similar users liked this m...
2300,6428,To Kill a Mockingbird,0.898,0.856,0.885,Recommended because similar users liked this m...
5845,16147,The Sopranos: Season 1,0.972,0.672,0.882,Recommended because similar users liked this m...
3918,10774,Taxi Driver,0.861,0.901,0.873,Recommended because similar users liked this m...



--- Top 5 Hybrid Recommendations for User: 1614320 ---


,movie_id,title,norm_svd,norm_cf,final_score,explanation
1681,4640,Rain Man,0.887,1.000,0.921,Recommended because similar users liked this m...
2572,7057,Lord of the Rings: The Two Towers: Extended Ed...,0.915,0.831,0.890,Recommended because similar users liked this m...
2634,7230,The Lord of the Rings: The Fellowship of the R...,0.907,0.824,0.882,Recommended because similar users liked this m...
5514,14961,Lord of the Rings: The Return of the King: Ext...,0.902,0.819,0.877,Recommended because similar users liked this m...
2350,6428,To Kill a Mockingbird,0.908,0.778,0.869,Recommended because similar users liked this m...



--- Top 5 Hybrid Recommendations for User: 52540 ---


,movie_id,title,norm_svd,norm_cf,final_score,explanation
538,1561,American Wedding,1.000,0.872,0.962,Recommended because similar users liked this m...
3855,11077,Meet the Fockers,0.982,0.875,0.950,Recommended because similar users liked this m...
1016,2992,The Rundown,0.972,0.807,0.922,Recommended because similar users liked this m...
3288,9471,The Butterfly Effect: Director's Cut,0.907,0.935,0.915,Recommended because similar users liked this m...
1780,5239,The Longest Yard,1.000,0.717,0.915,Recommended because similar users liked this m...



--- Top 5 Hybrid Recommendations for User: 1198785 ---


,movie_id,title,norm_svd,norm_cf,final_score,explanation
1467,4227,The Full Monty,0.735,0.903,0.785,Recommended because similar users liked this m...
4599,12778,Thelma & Louise: Special Edition,0.733,0.890,0.780,Recommended because similar users liked this m...
2636,7430,Six Feet Under: Season 1,0.846,0.589,0.769,Recommended because similar users liked this m...
998,2862,The Silence of the Lambs,0.664,1.000,0.764,Recommended because similar users liked this m...
1382,3962,Finding Nemo (Widescreen),0.681,0.953,0.763,Recommended because similar users liked this m...


## 4. Discussion

**Advantages of Hybrid Model:**
- **Best of Both Worlds:** We get the global predictive accuracy of SVD, combined with the local similarity matching of Item-CF.
- **Explainability Added to SVD:** SVD alone is a black box, but by injecting a 30% Item-CF weight, we guarantee that the final recommendations share *some* measurable similarity with the user's past history, allowing us to confidently provide the explanation string.

**Why Hybrid Outperforms Individual Models:**
- It prevents SVD from recommending mathematically sound but conceptually bizarre or irrelevant movies.
- It prevents Item-CF from repeatedly recommending the same top blockbusters (popularity bias) because the 70% SVD weight pulls the score towards true latent affinity.

**Remaining Limitations & Cold-Start:**
- **Computation:** We have to run inference on two models instead of one, doubling the processing time per user.
- **Cold-Start Problem:** If a brand new user registers with 0 ratings, Item-CF yields 0, and SVD simply defaults to the global mean. The hybrid formula will output identical default scores for everyone until the user rates at least a few movies. We still need a purely popularity-based fallback for day-one users.
